# Enhanced: split_dataset demo (Copilot-friendly)

This enhanced notebook adds a robust, documented `split_dataset` helper with an optional `random_seed` for reproducible splits and a short demo.

Notes:
- The function accepts a pandas DataFrame and returns `(train_df, test_df)`.
- If a variable `dataset_df` already exists in the notebook (from earlier cells), the demo will use it; otherwise a small sample DataFrame is created.
- This notebook is meant to be non-destructive: it does not overwrite your original `exercise-explore-your-data.ipynb` and is added as a companion in English.


In [ ]:
# Imports
import numpy as np
import pandas as pd
from typing import Tuple

def split_dataset(dataset: pd.DataFrame, test_ratio: float = 0.20, random_seed: int | None = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Split a pandas DataFrame into (train, test) using a random mask.

    Parameters
    ----------
    dataset : pd.DataFrame
        Input DataFrame to split. Rows are sampled without replacement into train/test.
    test_ratio : float, optional
        Fraction of rows to include in the test set (between 0 and 1). Default is 0.20.
    random_seed : int or None, optional
        Optional numpy seed for reproducible splits. If `None` the split is non-deterministic.

    Returns
    -------
    (train_df, test_df) : tuple[pd.DataFrame, pd.DataFrame]
        Train and test DataFrames with their indices reset.
    """
    if not isinstance(dataset, pd.DataFrame):
        raise TypeError('dataset must be a pandas DataFrame')
    if not (0.0 <= test_ratio <= 1.0):
        raise ValueError('test_ratio must be between 0.0 and 1.0')

    # Optionally set seed for reproducibility
    rng = np.random.RandomState(random_seed) if random_seed is not None else np.random

    # Create boolean mask for test set
    test_mask = rng.rand(len(dataset)) < test_ratio

    train_df = dataset.loc[~test_mask].reset_index(drop=True)
    test_df = dataset.loc[test_mask].reset_index(drop=True)

    return train_df, test_df


## Demo: run the split

The demo below will use an existing `dataset_df` variable if present (so it integrates with the original notebook). If not present, it will generate a small example DataFrame for demonstration purposes.


In [ ]:
# If the original notebook already defined `dataset_df`, use it; otherwise create a sample DataFrame.
try:
    _ = dataset_df  # reference from earlier cells in the original notebook
    demo_df = dataset_df.copy()
    print('Using dataset_df from the environment (original notebook).')
except NameError:
    demo_df = pd.DataFrame({"feature": range(100), "label": [x % 2 for x in range(100)]})
    print('No dataset_df found; using a generated demo DataFrame.')

train, test = split_dataset(demo_df, test_ratio=0.20, random_seed=42)
print(f"{len(train)} examples in training, {len(test)} examples in testing.")

# Show a small sample from each split
print('
Train sample:')
display(train.head())
print('
Test sample:')
display(test.head())


## Usage notes

- If you want stratified splits (to preserve label proportions) consider using scikit-learn's `train_test_split(..., stratify=...)`.
- This helper is intentionally simple and fast for small-to-medium DataFrames. For very large datasets prefer index-based sampling or using DataFrame.sample with a fixed random_state.
- You can import and re-use the function from `demo_split.py` added to the repository.
